# Federated Learning for Colon Cancer Histopathology Images with PIDL\n
\n
This notebook runs the **existing ResNet-18 + PIDL + federated learning pipeline** on the **colon cancer** subset of the Kaggle dataset `andrewmvd/lung-and-colon-cancer-histopathological-images`.\n
\n
- The **model, loss, and FL loop** are exactly the same as in the brain tumor project.\n
- We only change **which dataset path we use** and **how many clients** we simulate.\n
- Results (metrics, timings, confusion matrix, summary, and final model weights) are written into **dataset-specific directories**, so brain, colon, and later lung runs do not mix.\n

In [ ]:
# Mount Google Drive (optional, e.g., if you want to copy artifacts to Drive)\n
from google.colab import drive\n
drive.mount('/content/drive')\n

In [ ]:
# Clone the repository (use your GitHub URL) and go into project root\n
# If you already have the repo under /content, skip clone and set PROJECT_DIR accordingly.\n
import os\n
REPO_URL = "https://github.com/PulockDas/brain-tumor-classification-PIDL-FL.git"  # ← set your repo URL if different\n
PROJECT_DIR = "/content/brain-tumor-classification-PIDL-FL"\n
\n
if not os.path.isdir(PROJECT_DIR):\n
    !git clone --depth 1 {REPO_URL} {PROJECT_DIR}\n
%cd {PROJECT_DIR}\n

In [ ]:
# Install project and dependencies (including kagglehub for dataset download)\n
!cd {PROJECT_DIR} && pip install -e . kagglehub\n

## Download colon cancer dataset from Kaggle\n

In [ ]:
import os\n
import kagglehub\n
\n
# Download or reuse cached LC25000 lung & colon dataset\n
kaggle_path = kagglehub.dataset_download(\"andrewmvd/lung-and-colon-cancer-histopathological-images\")\n
print(\"Kaggle dataset base path:\", kaggle_path)\n
\n
# Expected structure inside kaggle_path:\n
# lung_colon_image_set/\n
#   colon_image_sets/\n
#       colon_aca/\n
#       colon_n/\n
#   lung_image_sets/\n
#       lung_aca/\n
#       lung_scc/\n
#       lung_n/\n
\n
colon_root = os.path.join(kaggle_path, \"lung_colon_image_set\", \"colon_image_sets\")\n
print(\"Colon dataset root:\", colon_root)\n
\n
if not os.path.isdir(colon_root):\n
    raise FileNotFoundError(f\"Colon dataset root not found: {colon_root}\")\n
\n
print(\"Colon classes:\", os.listdir(colon_root))\n

## Configuration\n

In [ ]:
import os\n
\n
# Paths\n
PROJECT_DIR = "/content/brain-tumor-classification-PIDL-FL"\n
# Base directory for logs outside the repo; we will later copy *only* important files back into the repo.\n
LOG_BASE_DIR = "/content/results_colon"\n
\n
# Dataset configuration\n
DATA_ROOT = colon_root           # Folder containing class subfolders (colon_aca, colon_n)\n
DATASET_NAME = \"colon_cancer\"  # Used only for naming result directories\n
\n
# Federated learning configuration\n
NUM_ROUNDS = 10                  # Number of FL rounds (change as needed)\n
LOCAL_EPOCHS = 5                 # Local epochs per round\n
NUM_CLIENTS = 3                  # e.g. 3, 5, 10, ...\n
\n
# Tag to distinguish multiple runs with the same setup\n
EXPERIMENT_TAG = f\"{DATASET_NAME}_{NUM_CLIENTS}clients_{NUM_ROUNDS}rds\"\n
\n
# This is where train_fl.py will actually write logs and the final model,\n
# given how it constructs dataset- and client-specific log directories.\n
RUN_LOG_DIR = os.path.join(\n
    LOG_BASE_DIR, DATASET_NAME, f\"{NUM_CLIENTS}_clients\", EXPERIMENT_TAG\n
)\n
\n
print(\"DATA_ROOT     :\", DATA_ROOT)\n
print(\"LOG_BASE_DIR  :\", LOG_BASE_DIR)\n
print(\"RUN_LOG_DIR   :\", RUN_LOG_DIR)\n
print(\"DATASET_NAME  :\", DATASET_NAME)\n
print(\"NUM_ROUNDS    :\", NUM_ROUNDS)\n
print(\"LOCAL_EPOCHS  :\", LOCAL_EPOCHS)\n
print(\"NUM_CLIENTS   :\", NUM_CLIENTS)\n
print(\"EXPERIMENT_TAG:\", EXPERIMENT_TAG)\n

## Run Federated Learning on Colon Cancer Dataset\n

In [ ]:
import os\n
import subprocess\n
\n
os.makedirs(LOG_BASE_DIR, exist_ok=True)\n
\n
cmd = [\n
    \"python\", \"train_fl.py\",\n
    \"--data-root\", DATA_ROOT,\n
    \"--dataset-name\", DATASET_NAME,\n
    \"--num-clients\", str(NUM_CLIENTS),\n
    \"--num-rounds\", str(NUM_ROUNDS),\n
    \"--local-epochs\", str(LOCAL_EPOCHS),\n
    \"--log-dir\", LOG_BASE_DIR,\n
    \"--experiment-tag\", EXPERIMENT_TAG,\n
]\n
print("Running command:\n", " ".join(cmd))
result = subprocess.run(cmd, check=False)
print("Return code:", result.returncode)
print("\nExpected run log dir:", RUN_LOG_DIR)
print(\"The following files should be there after training finishes:\")\n
print(\"  fl_rounds.csv, fl_clients.csv, fl_eval.json, config.json, fl_summary.json, final_model.pth\")\n

## Plot Results (accuracy, F1, losses, timings, confusion matrix)\n

In [ ]:
import json\n
import os\n
import pandas as pd\n
import matplotlib.pyplot as plt\n
import numpy as np\n
\n
log_dir = RUN_LOG_DIR\n
rounds_path = os.path.join(log_dir, \"fl_rounds.csv\")\n
clients_path = os.path.join(log_dir, \"fl_clients.csv\")\n
eval_path = os.path.join(log_dir, \"fl_eval.json\")\n
summary_path = os.path.join(log_dir, \"fl_summary.json\")\n
\n
if not os.path.isfile(rounds_path) or not os.path.isfile(clients_path):\n
    print(\"No results yet. Run the federated learning cell first.\")\n
else:\n
    rounds_df = pd.read_csv(rounds_path)\n
    clients_df = pd.read_csv(clients_path)\n
\n
    # Global test accuracy over rounds\n
    plt.figure(figsize=(10, 6))\n
    plt.plot(rounds_df[\"round\"], rounds_df[\"global_test_acc\"], marker=\"o\", linewidth=2)\n
    plt.xlabel(\"Round\")\n
    plt.ylabel(\"Test Accuracy (%)\")\n
    plt.title(\"Global Test Accuracy over FL Rounds (Colon Cancer)\")\n
    plt.grid(True)\n
    plt.show()\n
\n
    # F1 macro over rounds (if present)\n
    if \"f1_macro\" in rounds_df.columns:\n
        plt.figure(figsize=(10, 6))\n
        plt.plot(rounds_df[\"round\"], rounds_df[\"f1_macro\"], marker=\"o\", linewidth=2, color=\"green\")\n
        plt.xlabel(\"Round\")\n
        plt.ylabel(\"F1 (macro)\")\n
        plt.title(\"Global Test F1 (macro) over FL Rounds (Colon Cancer)\")\n
        plt.grid(True)\n
        plt.show()\n
\n
    # Inference and training time per round (if present)\n
    if \"inference_time_sec\" in rounds_df.columns and \"training_time_sec\" in rounds_df.columns:\n
        fig, ax = plt.subplots(figsize=(10, 6))\n
        ax.plot(rounds_df[\"round\"], rounds_df[\"inference_time_sec\"], marker=\"o\", label=\"Inference time (s)\", linewidth=2)\n
        ax.plot(rounds_df[\"round\"], rounds_df[\"training_time_sec\"], marker=\"s\", label=\"Training time (s)\", linewidth=2)\n
        ax.set_xlabel(\"Round\")\n
        ax.set_ylabel(\"Time (s)\")\n
        ax.set_title(\"Inference & Training Time per Round (Colon Cancer)\")\n
        ax.legend()\n
        ax.grid(True)\n
        plt.tight_layout()\n
        plt.show()\n
\n
    # Client training accuracies\n
    plt.figure(figsize=(12, 6))\n
    for cid in clients_df[\"client_id\"].unique():\n
        d = clients_df[clients_df[\"client_id\"] == cid]\n
        plt.plot(d[\"round\"], d[\"train_acc\"], marker=\"o\", label=f\"Client {cid}\", linewidth=2)\n
    plt.xlabel(\"Round\")\n
    plt.ylabel(\"Training Accuracy (%)\")\n
    plt.title(\"Client Training Accuracies over FL Rounds (Colon Cancer)\")\n
    plt.legend()\n
    plt.grid(True)\n
    plt.show()\n
\n
    # Losses\n
    plt.figure(figsize=(12, 6))\n
    plt.subplot(1, 2, 1)\n
    plt.plot(rounds_df[\"round\"], rounds_df[\"global_test_loss\"], marker=\"o\", linewidth=2)\n
    plt.xlabel(\"Round\")\n
    plt.ylabel(\"Test Loss\")\n
    plt.title(\"Global Test Loss (Colon Cancer)\")\n
    plt.grid(True)\n
\n
    plt.subplot(1, 2, 2)\n
    for cid in clients_df[\"client_id\"].unique():\n
        d = clients_df[clients_df[\"client_id\"] == cid]\n
        plt.plot(d[\"round\"], d[\"train_loss\"], marker=\"o\", label=f\"Client {cid}\", linewidth=2)\n
    plt.xlabel(\"Round\")\n
    plt.ylabel(\"Training Loss\")\n
    plt.title(\"Client Training Losses (Colon Cancer)\")\n
    plt.legend()\n
    plt.grid(True)\n
    plt.tight_layout()\n
    plt.show()\n
\n
    # Final confusion matrix (from fl_eval.json, last round)\n
    if os.path.isfile(eval_path):\n
        with open(eval_path) as f:\n
            ev = json.load(f)\n
        rounds_ev = ev.get(\"rounds\", [])\n
        class_names = ev.get(\"class_names\")\n
        if not class_names and rounds_ev and \"confusion_matrix\" in rounds_ev[-1]:\n
            # Fallback generic names if class names are missing\n
            cm_tmp = np.array(rounds_ev[-1][\"confusion_matrix\"])\n
            class_names = [f\"C{i}\" for i in range(cm_tmp.shape[0])]\n
\n
        if rounds_ev and \"confusion_matrix\" in rounds_ev[-1]:\n
            cm = np.array(rounds_ev[-1][\"confusion_matrix\"])\n
            plt.figure(figsize=(8, 6))\n
            plt.imshow(cm, interpolation=\"nearest\", cmap=\"Blues\")\n
            plt.colorbar()\n
            if class_names is not None:\n
                plt.xticks(np.arange(len(class_names)), class_names, rotation=45, ha=\"right\")\n
                plt.yticks(np.arange(len(class_names)), class_names)\n
            plt.xlabel(\"Predicted\")\n
            plt.ylabel(\"True\")\n
            plt.title(\"Confusion Matrix (final round, Colon Cancer)\")\n
            for i in range(cm.shape[0]):\n
                for j in range(cm.shape[1]):\n
                    plt.text(j, i, int(cm[i, j]), ha=\"center\", va=\"center\",\n
                             color=\"black\" if cm[i, j] < cm.max() / 2 else \"white\")\n
            plt.tight_layout()\n
            plt.show()\n
\n
    if os.path.isfile(summary_path):\n
        with open(summary_path) as f:\n
            s = json.load(f)\n
        print(\"Summary:\", json.dumps(s, indent=2))\n

## Copy important result files (including trained model) back into the repo and push to GitHub\n
\n
This cell copies only the **essential artifacts** from the run-specific directory into the repository under `results/`,\n
then stages, commits, and (optionally) pushes them to GitHub.\n
\n
Artifacts copied:\n
- `fl_rounds.csv`\n
- `fl_clients.csv`\n
- `fl_eval.json`\n
- `config.json`\n
- `fl_summary.json`\n
- `final_model.pth` (final trained model weights, so you can re-use them without retraining)\n

In [ ]:
import os\n
import shutil\n
import subprocess\n
\n
PROJECT_DIR = "/content/brain-tumor-classification-PIDL-FL"\n
\n
# Run-specific log directory (source) and repo destination directory\n
SRC_DIR = RUN_LOG_DIR\n
DEST_DIR = os.path.join(\n
    PROJECT_DIR, \"results\", DATASET_NAME, f\"{NUM_CLIENTS}_clients\", EXPERIMENT_TAG\n
)\n
\n
FILES = [\n
    \"fl_rounds.csv\",\n
    \"fl_clients.csv\",\n
    \"fl_eval.json\",\n
    \"config.json\",\n
    \"fl_summary.json\",\n
    \"final_model.pth\",\n
]\n
\n
os.makedirs(DEST_DIR, exist_ok=True)\n
copied = []\n
for f in FILES:\n
    src = os.path.join(SRC_DIR, f)\n
    dst = os.path.join(DEST_DIR, f)\n
    if os.path.isfile(src):\n
        shutil.copy2(src, dst)\n
        copied.append(os.path.relpath(dst, PROJECT_DIR))\n
    else:\n
        print(f\"Skipping {f} (not found in {SRC_DIR})\")\n
\n
if not copied:\n
    print(\"No result files to push. Run the FL cell first.\")\n
else:\n
    print(\"Copied to repo (under results/):\")\n
    for rel in copied:\n
        print(\"  ", rel)\n
\n
    # Stage copied files\n
    for rel in copied:\n
        subprocess.run([\"git\", \"add\", rel], cwd=PROJECT_DIR, check=True)\n
    subprocess.run([\"git\", \"status\"], cwd=PROJECT_DIR, check=True)\n
\n
    # Commit (only if there is something to commit)\n
    commit_res = subprocess.run(\n
        [\"git\", \"commit\", \"-m\", \"Add colon cancer FL result artifacts\"],\n
        cwd=PROJECT_DIR,\n
        text=True,\n
        capture_output=True,\n
    )\n
    if commit_res.returncode == 0:\n
        print(commit_res.stdout)\n
    else:\n
        # 1 means \"nothing to commit\" for many git versions; print message and continue\n
        print(commit_res.stdout or commit_res.stderr)\n
\n
    # Read token from environment (recommended: Colab Secrets -> env var)\n
    token = os.environ.get(\"GITHUB_TOKEN\")\n
    if not token:\n
        print(\"\\nNot pushing: missing GITHUB_TOKEN (Colab can't prompt for GitHub credentials).\")\n
        print(\"Set GITHUB_TOKEN in the environment (e.g., Colab Secrets) and rerun this cell if you want to push.\")\n
    else:\n
        origin_url = subprocess.check_output([\"git\", \"remote\", \"get-url\", \"origin\"], cwd=PROJECT_DIR, text=True).strip()\n
        if origin_url.startswith(\"https://\"):\n
            push_url = origin_url.replace(\"https://\", f\"https://{token}@\")\n
            push_res = subprocess.run([\"git\", \"push\", push_url, \"HEAD\"], cwd=PROJECT_DIR)\n
        else:\n
            # SSH remote\n
            push_res = subprocess.run([\"git\", \"push\", \"origin\", \"HEAD\"], cwd=PROJECT_DIR)\n
\n
        if push_res.returncode == 0:\n
            print(\"\\nPushed to GitHub.\")\n
        else:\n
            raise SystemExit(\"git push failed (see output above).\")\n